In [1]:
!pip install captum grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 95.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 43.6 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=d695e669ac97964f90e03a1992f48e6d23a1f5981ee198c0600b34478cce2a6e
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from captum.attr import Saliency, IntegratedGradients, GuidedBackprop, LRP
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

IMAGE_PATH = "/content/n02051845_pelican.JPEG"
TARGET_CLASS = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_PATH = "attribution_comparison2.png"

weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights).to(DEVICE).eval()

preprocess = weights.transforms()
vis_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def load_image(path: str):
    img = Image.open(path).convert("RGB")
    img_vis = vis_transform(img)
    img_np = np.array(img_vis).astype(np.float32) / 255.0
    inp = preprocess(img).unsqueeze(0).to(DEVICE)
    return img_np, inp


def get_target(inp: torch.Tensor):
    with torch.no_grad():
        logits = model(inp)
        pred = logits.argmax(dim=1).item()
    return pred if TARGET_CLASS is None else TARGET_CLASS


def normalize_map(x: np.ndarray):
    x = x.astype(np.float32)
    x = x - x.min()
    denom = x.max() - x.min()
    if denom < 1e-8:
        return np.zeros_like(x)
    return x / denom


def tensor_attr_to_heatmap(attr: torch.Tensor):
    arr = attr.detach().cpu().numpy()[0]
    if arr.ndim == 3:
        arr = np.abs(arr).mean(axis=0)
    else:
        arr = np.abs(arr[0])
    return normalize_map(arr)


def overlay_heatmap(img_rgb: np.ndarray, heatmap: np.ndarray, cmap="jet", alpha=0.45):
    cmap_fn = plt.get_cmap(cmap)
    color = cmap_fn(heatmap)[..., :3]
    out = (1 - alpha) * img_rgb + alpha * color
    return np.clip(out, 0, 1)


def run_saliency(inp: torch.Tensor, target: int):
    x = inp.clone().detach().requires_grad_(True)
    sal = Saliency(model)
    attr = sal.attribute(x, target=target)
    return tensor_attr_to_heatmap(attr)


def run_integrated_gradients(inp: torch.Tensor, target: int):
    x = inp.clone().detach().requires_grad_(True)
    ig = IntegratedGradients(model)
    baseline = torch.zeros_like(x)
    attr = ig.attribute(x, baselines=baseline, target=target, n_steps=50)
    return tensor_attr_to_heatmap(attr)


def run_guided_backprop(inp: torch.Tensor, target: int):
    x = inp.clone().detach().requires_grad_(True)
    gbp = GuidedBackprop(model)
    attr = gbp.attribute(x, target=target)
    return tensor_attr_to_heatmap(attr)


def run_lrp(inp: torch.Tensor, target: int):
    x = inp.clone().detach().requires_grad_(True)
    lrp = LRP(model)
    attr = lrp.attribute(x, target=target)
    return tensor_attr_to_heatmap(attr)


def run_gradcam(img_rgb: np.ndarray, inp: torch.Tensor, target: int):
    target_layers = [model.layer4[-1]]
    targets = [ClassifierOutputTarget(target)]
    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=inp, targets=targets)[0]
    return show_cam_on_image(img_rgb, grayscale_cam, use_rgb=True)

def main():
    img_rgb, inp = load_image(IMAGE_PATH)
    target = get_target(inp)

    categories = weights.meta["categories"]
    print(f"Target class index: {target}")
    print(f"Target class name : {categories[target]}")

    results = {}
    results["Original"] = img_rgb
    results["Saliency"] = overlay_heatmap(img_rgb, run_saliency(inp, target))
    results["Integrated\nGradients"] = overlay_heatmap(img_rgb, run_integrated_gradients(inp, target))
    results["Guided\nBackprop"] = overlay_heatmap(img_rgb, run_guided_backprop(inp, target))
    results["LRP"] = overlay_heatmap(img_rgb, run_lrp(inp, target))
    results["Grad-CAM"] = run_gradcam(img_rgb, inp, target)

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()

    for ax, (title, image) in zip(axes, results.items()):
        ax.imshow(image)
        ax.set_title(title, fontsize=12)
        ax.axis("off")

    for ax in axes[len(results):]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(OUT_PATH, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved figure to {OUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
main()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("attribution_comparison.png")
plt.imshow(img)
plt.axis("off")